In [1]:
# statsbomb_shots.py
import json
import math
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
import os

# Go one level higher
os.chdir("../..")

# Check current directory
print(os.getcwd())

/Users/rodrigo/football-data-analytics


In [3]:
# CONFIG: point this to your local copy of StatsBomb open-data
OPEN_DATA_DIR = Path("open-data/data")  # adjust if different
EVENTS_DIR = OPEN_DATA_DIR / "events"

# Geometry for StatsBomb coordinates:
# StatsBomb uses a 120x80 pitch with the attacking goal on the right at x=120, centered y=40
GOAL_X = 120.0
GOAL_CENTER_Y = 40.0
LEFT_POST = (GOAL_X, GOAL_CENTER_Y - 4.0)   # post positions assuming 8 unit goal width (36 and 44)
RIGHT_POST = (GOAL_X, GOAL_CENTER_Y + 4.0)


In [4]:
def load_events_json(path: Path):
    with open(path, "r", encoding="utf8") as f:
        return json.load(f)

def euclid(a, b):
    return math.hypot(a[0]-b[0], a[1]-b[1])

def shot_distance(x, y):
    return euclid((x,y), (GOAL_X, GOAL_CENTER_Y))

def shot_angle(x, y):
    # angle subtended by the two goal posts at shot location (radians)
    vx1 = (LEFT_POST[0] - x, LEFT_POST[1] - y)
    vx2 = (RIGHT_POST[0] - x, RIGHT_POST[1] - y)
    # angle between vectors
    dot = vx1[0]*vx2[0] + vx1[1]*vx2[1]
    mag1 = math.hypot(vx1[0], vx1[1])
    mag2 = math.hypot(vx2[0], vx2[1])
    if mag1 * mag2 == 0:
        return 0.0
    cosang = max(-1.0, min(1.0, dot / (mag1 * mag2)))
    return math.acos(cosang)  # radians

def parse_statsbomb_events_file(path: Path):
    events = load_events_json(path)
    shots = []
    for ev in events:
        if ev.get("type", {}).get("name") == "Shot":
            shot = {}
            shot["match_id"] = ev.get("match_id")
            shot["event_id"] = ev.get("id")
            shot["player_id"] = ev.get("player", {}).get("id")
            shot["player_name"] = ev.get("player", {}).get("name")
            shot["team_id"] = ev.get("team", {}).get("id")
            shot["team_name"] = ev.get("team", {}).get("name")
            shot["minute"] = ev.get("minute")
            shot["second"] = ev.get("second")
            # outcome/subtype
            shot["shot_outcome"] = ev.get("shot", {}).get("outcome", {}).get("name")
            shot["shot_body_part"] = ev.get("shot", {}).get("body_part", {}).get("name")
            shot["shot_technique"] = ev.get("shot", {}).get("technique", {}).get("name")
            # location
            loc = ev.get("location", [None, None])
            shot["x"] = float(loc[0]) if loc[0] is not None else None
            shot["y"] = float(loc[1]) if loc[1] is not None else None
            if shot["x"] is not None and shot["y"] is not None:
                shot["dist_to_goal"] = shot_distance(shot["x"], shot["y"])
                shot["angle_to_goal_rad"] = shot_angle(shot["x"], shot["y"])
                shot["angle_to_goal_deg"] = math.degrees(shot["angle_to_goal_rad"])
            else:
                shot["dist_to_goal"] = None
                shot["angle_to_goal_rad"] = None
                shot["angle_to_goal_deg"] = None

            shots.append(shot)
    return pd.DataFrame(shots)

def collect_all_shots(events_dir: Path):
    dfs = []
    for path in events_dir.glob("**/*.json"):
        try:
            df = parse_statsbomb_events_file(path)
            if not df.empty:
                dfs.append(df)
        except Exception as e:
            print(f"failed to parse {path}: {e}")
    if dfs:
        return pd.concat(dfs, ignore_index=True)
    else:
        return pd.DataFrame()

In [ ]:
shots_df = collect_all_shots(EVENTS_DIR)

In [ ]:
shots_df.shape

(0, 0)